# Model inventory table

Filter a model-independent element table and export the current result to CSV.

This sample uses only standard SysML v2 concepts and reads the model through `memo_model.py`, compiled by the project's selected compiler.

In [ ]:
import sys
from pathlib import Path
from collections import Counter, defaultdict

# memo_model.py sits in the project's analysis/ folder, above this notebook.
sys.path.insert(0, str(next(d for d in (Path.cwd(), *Path.cwd().parents) if (d / 'memo_model.py').is_file())))
import memo_model

# Compiled by the project's selected compiler (toolchain.compiler), as every
# MEMO command is: change the setting and this notebook follows.
model = memo_model.load()
diagnostics = model.diagnostics
SYSML_FILES = model.documents
print(f'Loaded {len(SYSML_FILES)} SysML files, compiled by {model.compiler} {model.compiler_version or ""}')

In [ ]:
from html import escape
from IPython.display import HTML, display

KIND_FILTER = ''   # Example: 'PartUsage' or 'Requirement'
NAME_FILTER = ''   # Case-insensitive name fragment
ROW_LIMIT = 250

semantic_elements = (
    model.elements('Usage', include_subtypes=True)
    + model.elements('Definition', include_subtypes=True)
)

def text(value):
    return str(value) if value else ''

records = []
for item in semantic_elements:
    kind = item.type
    name = text(item.name or item.declared_name)
    qualified_name = text(item.qualified_name)
    owner = getattr(item, 'owner', None)
    owner_name = text(getattr(owner, 'qualified_name', None) or getattr(owner, 'name', None))
    if KIND_FILTER and KIND_FILTER.casefold() not in kind.casefold():
        continue
    if NAME_FILTER and NAME_FILTER.casefold() not in (name + ' ' + qualified_name).casefold():
        continue
    records.append({'kind': kind, 'name': name, 'qualified_name': qualified_name, 'owner': owner_name})

records.sort(key=lambda row: (row['kind'], row['qualified_name'], row['name']))
visible = records[:ROW_LIMIT]
rows = ''.join(
    '<tr>' + ''.join(f'<td>{escape(row[key])}</td>' for key in ('kind', 'name', 'qualified_name', 'owner')) + '</tr>'
    for row in visible
)
display(HTML(f'''<p><strong>{len(records):,}</strong> matching elements; showing {len(visible):,}.</p>
<div style="max-height:620px;overflow:auto"><table style="font-size:12px"><thead style="position:sticky;top:0;background:white"><tr><th>Kind</th><th>Name</th><th>Qualified name</th><th>Owner</th></tr></thead><tbody>{rows}</tbody></table></div>'''))

In [ ]:
import csv

OUTPUT = Path('model-inventory.csv')
with OUTPUT.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=['kind', 'name', 'qualified_name', 'owner'])
    writer.writeheader()
    writer.writerows(records)
print(f'Exported {len(records):,} rows to {OUTPUT.resolve()}')